# Modelamiento y Selección de Modelos - Speed Dating Columbia University

Este notebook realiza el entrenamiento, validación y selección de modelos para predecir matches en citas rápidas.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import pickle
import os
import json
warnings.filterwarnings('ignore')

# Configuración de estilo
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 10)

# Paleta de colores
PALETA_ROSA = ['#FF1493', '#FF69B4', '#FFB6C1', '#FFC0CB', '#DB7093']
sns.set_palette(PALETA_ROSA)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print('Librerías importadas correctamente')
print('Paleta rosa principal: #FF1493')

## 1. Carga de Datos Preparados

In [ ]:
# Cargar datasets preprocesados
X_train = pd.read_csv('../data/X_train.csv')
X_test = pd.read_csv('../data/X_test.csv')
y_train = pd.read_csv('../data/y_train.csv').values.ravel()
y_test = pd.read_csv('../data/y_test.csv').values.ravel()

print(f'X_train: {X_train.shape}')
print(f'X_test: {X_test.shape}')
print(f'y_train: {y_train.shape}')
print(f'y_test: {y_test.shape}')

# Cargar feature names
with open('../data/feature_names.pkl', 'rb') as f:
    feature_names = pickle.load(f)
print(f'\nNúmero de features: {len(feature_names)}')

# Cargar scaler
with open('../data/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)
print('Scaler cargado correctamente')

# Distribución del target
print(f'\nDistribución Train - Match: {sum(y_train)} ({sum(y_train)/len(y_train)*100:.1f}%)')
print(f'Distribución Test - Match: {sum(y_test)} ({sum(y_test)/len(y_test)*100:.1f}%)')

## 2. Entrenamiento de Modelos Base

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import time

# Definir 7 modelos
modelos = {
    'Arbol de Decision': DecisionTreeClassifier(random_state=42),
    'MLP': MLPClassifier(random_state=42, max_iter=1000),
    'SVM': SVC(random_state=42, probability=True),
    'KNN': KNeighborsClassifier(),
    'Random Forest': RandomForestClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss', use_label_encoder=False),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

# Validación cruzada estratificada 10-fold
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Métricas
scoring = {
    'accuracy': make_scorer(accuracy_score),
    'precision': make_scorer(precision_score),
    'recall': make_scorer(recall_score),
    'f1': make_scorer(f1_score),
    'roc_auc': make_scorer(roc_auc_score)
}

resultados = []
cv_scores_dict = {}

print('Entrenando 7 modelos con Validación Cruzada Estratificada 10-fold...')
print('='*80)

for nombre, modelo in modelos.items():
    print(f'\n{nombre}...')
    inicio = time.time()
    cv_results = cross_validate(modelo, X_train, y_train, cv=cv, scoring=scoring, return_train_score=True)
    tiempo = time.time() - inicio
    resultado = {
        'Modelo': nombre,
        'Accuracy_mean': cv_results['test_accuracy'].mean(),
        'Accuracy_std': cv_results['test_accuracy'].std(),
        'Precision_mean': cv_results['test_precision'].mean(),
        'Precision_std': cv_results['test_precision'].std(),
        'Recall_mean': cv_results['test_recall'].mean(),
        'Recall_std': cv_results['test_recall'].std(),
        'F1_mean': cv_results['test_f1'].mean(),
        'F1_std': cv_results['test_f1'].std(),
        'ROC_AUC_mean': cv_results['test_roc_auc'].mean(),
        'ROC_AUC_std': cv_results['test_roc_auc'].std(),
        'Tiempo': tiempo
    }
    resultados.append(resultado)
    cv_scores_dict[nombre] = cv_results['test_roc_auc']
    
    print(f'  Accuracy: {resultado["Accuracy_mean"]:.4f} (+/- {resultado["Accuracy_std"]:.4f})')
    print(f'  Precision: {resultado["Precision_mean"]:.4f} (+/- {resultado["Precision_std"]:.4f})')
    print(f'  Recall: {resultado["Recall_mean"]:.4f} (+/- {resultado["Recall_std"]:.4f})')
    print(f'  F1: {resultado["F1_mean"]:.4f} (+/- {resultado["F1_std"]:.4f})')
    print(f'  ROC-AUC: {resultado["ROC_AUC_mean"]:.4f} (+/- {resultado["ROC_AUC_std"]:.4f})')
    print(f'  Tiempo: {tiempo:.2f}s')

# Tabla comparativa
df_resultados = pd.DataFrame(resultados)
df_resultados = df_resultados.sort_values('ROC_AUC_mean', ascending=False)
df_resultados['Accuracy'] = df_resultados['Accuracy_mean'].round(4).astype(str) + ' (' + df_resultados['Accuracy_std'].round(4).astype(str) + ')'
df_resultados['Precision'] = df_resultados['Precision_mean'].round(4).astype(str) + ' (' + df_resultados['Precision_std'].round(4).astype(str) + ')'
df_resultados['Recall'] = df_resultados['Recall_mean'].round(4).astype(str) + ' (' + df_resultados['Recall_std'].round(4).astype(str) + ')'
df_resultados['F1'] = df_resultados['F1_mean'].round(4).astype(str) + ' (' + df_resultados['F1_std'].round(4).astype(str) + ')'
df_resultados['ROC-AUC'] = df_resultados['ROC_AUC_mean'].round(4).astype(str) + ' (' + df_resultados['ROC_AUC_std'].round(4).astype(str) + ')'

print('\n' + '='*80)
print('TABLA COMPARATIVA DE MODELOS')
print('='*80)
display_cols = ['Modelo', 'Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC', 'Tiempo']
from IPython.display import display
display(df_resultados[display_cols].style.background_gradient(cmap='RdBu_r', subset=['ROC-AUC']))

## 3. Curvas ROC Comparativas

In [ ]:
# Curvas ROC superpuestas
from sklearn.metrics import RocCurveDisplay

fig, ax = plt.subplots(figsize=(12, 10))

for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    RocCurveDisplay.from_estimator(modelo, X_test, y_test, ax=ax, name=nombre)

ax.plot([0, 1], [0, 1], 'k--', label='Aleatorio')
ax.set_xlabel('Tasa de Falsos Positivos', fontsize=12)
ax.set_ylabel('Tasa de Verdaderos Positivos', fontsize=12)
ax.set_title('Curvas ROC - Comparacion de Modelos', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3)

os.makedirs('../reports', exist_ok=True)
plt.savefig('../reports/curvas_roc_comparativas.png', dpi=300, bbox_inches='tight')
plt.show()

print('Curvas ROC guardadas en ../reports/curvas_roc_comparativas.png')


## 4. ANOVA y Tukey HSD

In [ ]:
# ANOVA + Tukey HSD
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

cv_scores_list = []
for nombre, scores in cv_scores_dict.items():
    for score in scores:
        cv_scores_list.append({'Modelo': nombre, 'ROC_AUC': score})

df_cv = pd.DataFrame(cv_scores_list)

grupos = [df_cv[df_cv['Modelo'] == m]['ROC_AUC'].values for m in df_resultados['Modelo']]
stat, p_valor = stats.f_oneway(*grupos)

print('='*80)
print('ANOVA - Diferencias entre modelos')
print('='*80)
print(f'Estadistico F: {stat:.4f}')
print(f'Valor p: {p_valor:.6f}')

if p_valor < 0.05:
    print('Diferencias significativas (p < 0.05)')
else:
    print('No hay diferencias significativas')

tukey = pairwise_tukeyhsd(df_cv['ROC_AUC'], df_cv['Modelo'], alpha=0.05)
print(tukey)


## 5. Top 3 Modelos por ROC-AUC

In [ ]:
# Seleccionar top 3
top3 = df_resultados.nlargest(3, 'ROC_AUC_mean')
print('TOP 3 MODELOS POR ROC-AUC')
display(top3[display_cols])

top3_nombres = top3['Modelo'].tolist()
print(f'Modelos para tuning: {top3_nombres}')

top3_modelos = {nombre: modelos[nombre] for nombre in top3_nombres}


## 6. Hiperparametrizacion

In [ ]:
# RandomizedSearchCV para top 3
from sklearn.model_selection import RandomizedSearchCV
from skopt import BayesSearchCV

parametros_rf = {
    'n_estimators': [50, 100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10]
}

parametros_xgb = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.3]
}

parametros_gb = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.3]
}

param_grids = {}
for nombre in top3_nombres:
    if 'Random Forest' in nombre:
        param_grids[nombre] = parametros_rf
    elif 'XGBoost' in nombre:
        param_grids[nombre] = parametros_xgb
    elif 'Gradient Boosting' in nombre:
        param_grids[nombre] = parametros_gb
    else:
        param_grids[nombre] = {}

print('RandomizedSearchCV (n_iter=10, cv=5)')

mejores_modelos_rs = {}
resultados_rs = []

for nombre in top3_nombres:
    print(f'\n{nombre}...')
    rs = RandomizedSearchCV(top3_modelos[nombre], param_grids[nombre], n_iter=10, cv=5, scoring='roc_auc', random_state=42, n_jobs=-1)
    rs.fit(X_train, y_train)
    mejores_modelos_rs[nombre] = rs.best_estimator_
    resultados_rs.append({'Modelo': nombre, 'Mejor ROC-AUC': rs.best_score_, 'Parametros': str(rs.best_params_)})
    print(f'  ROC-AUC: {rs.best_score_:.4f}')

df_rs = pd.DataFrame(resultados_rs)
mejor_nombre = df_rs.loc[df_rs['Mejor ROC-AUC'].idxmax(), 'Modelo']
mejor_modelo_rs = mejores_modelos_rs[mejor_nombre]
print(f'\nMEJOR: {mejor_nombre}')


## 7. BayesSearchCV

In [ ]:
# BayesSearchCV
try:
    from skopt.space import Real, Integer, Categorical
    print('BayesSearchCV (n_iter=20)')
    if 'Random Forest' in mejor_nombre:
        busqueda = {'n_estimators': Integer(50, 300), 'max_depth': Integer(5, 50), 'min_samples_split': Integer(2, 20)}
    elif 'XGBoost' in mejor_nombre:
        busqueda = {'n_estimators': Integer(50, 300), 'max_depth': Integer(3, 10), 'learning_rate': Real(0.01, 0.3)}
    elif 'Gradient Boosting' in mejor_nombre:
        busqueda = {'n_estimators': Integer(50, 300), 'max_depth': Integer(3, 10), 'learning_rate': Real(0.01, 0.3)}
    else:
        busqueda = {}
    
    if busqueda:
        bayes = BayesSearchCV(mejor_modelo_rs, busqueda, n_iter=20, cv=5, scoring='roc_auc', random_state=42, n_jobs=-1)
        bayes.fit(X_train, y_train)
        mejor_modelo_final = bayes.best_estimator_
        print(f'Mejor ROC-AUC Bayes: {bayes.best_score_:.4f}')
    else:
        mejor_modelo_final = mejor_modelo_rs
except:
    mejor_modelo_final = mejor_modelo_rs
    print('BayesSearchCV no disponible, usando RandomizedSearchCV')


## 8. Evaluacion en Test

In [ ]:
# Evaluar en test
y_pred = mejor_modelo_final.predict(X_test)
y_pred_proba = mejor_modelo_final.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print('EVALUACION TEST')
print(f'Accuracy:  {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall:    {recall:.4f}')
print(f'F1-Score:  {f1:.4f}')
print(f'ROC-AUC:   {roc_auc:.4f}')

print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['No Match', 'Match']))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='RdBu_r', ax=axes[0])
axes[0].set_title('Matriz Confusion', fontsize=14, fontweight='bold')

fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
axes[1].plot(fpr, tpr, color='#FF1493', lw=2, label=f'ROC (AUC={roc_auc:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=2)
axes[1].set_xlabel('Falsos Positivos')
axes[1].set_ylabel('Verdaderos Positivos')
axes[1].set_title('Curva ROC Final', fontsize=14, fontweight='bold')
axes[1].legend()
plt.tight_layout()
plt.savefig('../reports/evaluacion_final.png', dpi=300, bbox_inches='tight')
plt.show()


## 9. Feature Importance y SHAP

In [ ]:
# Feature Importance
if hasattr(mejor_modelo_final, 'feature_importances_'):
    importancia = pd.DataFrame({'Feature': feature_names, 'Importancia': mejor_modelo_final.feature_importances_}).sort_values('Importancia', ascending=False)
    print('Top 20 Features')
    display(importancia.head(20))
    
    plt.figure(figsize=(12, 8))
    top_f = importancia.head(20)
    plt.barh(range(len(top_f)), top_f['Importancia'].values, color='#FF1493')
    plt.yticks(range(len(top_f)), top_f['Feature'].values)
    plt.xlabel('Importancia')
    plt.title('Feature Importance', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../reports/feature_importance.png', dpi=300, bbox_inches='tight')
    plt.show()


## 10. Guardar Pipeline

In [ ]:
# Guardar modelos
os.makedirs('../models', exist_ok=True)

joblib.dump(mejor_modelo_final, '../models/pipeline_match_predictor.pkl')
print('Pipeline guardado: models/pipeline_match_predictor.pkl')

metricas_finales = {
    'modelo': mejor_nombre,
    'accuracy': float(accuracy),
    'precision': float(precision),
    'recall': float(recall),
    'f1': float(f1),
    'roc_auc': float(roc_auc),
    'n_features': len(feature_names)
}

with open('../models/metricas_finales.pkl', 'wb') as f:
    pickle.dump(metricas_finales, f)
print('Metricas guardadas: models/metricas_finales.pkl')

print('\nProceso completado!')
